In [ ]:
!pip install waymo-open-dataset-tf-2-12-0 --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 19.4 MB/s eta 0:00:00


In [ ]:
from waymo_open_dataset.protos import scenario_pb2
import tensorflow as tf
import pandas as pd
import numpy as np
from waymo_open_dataset.protos import scenario_pb2








In [ ]:

# ============================================================
# STEP 1: Read the TFRecord file (Topic 1, Slide 4)
# ============================================================
filename = "uncompressed_scenario_validation_validation.tfrecord-00000-of-00150"

dataset = tf.data.TFRecordDataset(filename, compression_type="")

# ============================================================
# STEP 2: Generator to decode scenarios one at a time (Topic 1, Slide 5)
# ============================================================
def scenario_generator(dataset):
    for raw_record in dataset:
        scenario = scenario_pb2.Scenario()
        scenario.ParseFromString(raw_record.numpy())
        yield scenario

# ============================================================
# STEP 3: Flatten scenario -> rows (Topic 1, Slide 2)
# ============================================================
def scenario_to_rows(scenario):
    rows = []
    for track in scenario.tracks:
        for i, state in enumerate(track.states):
            if not state.valid:
                continue
            rows.append({
                "scenario_id": scenario.scenario_id,
                "agent_id": track.id,
                "agent_type": track.object_type,
                "timestep": i,
                "x": state.center_x,
                "y": state.center_y,
                "velocity_x": state.velocity_x,
                "velocity_y": state.velocity_y,
                "heading": state.heading,
            })
    return rows

# ============================================================
# STEP 4: Build the flat table — limit to first N scenarios for speed
# ============================================================
N_SCENARIOS = 5  # keep small while testing; raise later

all_rows = []
for i, scenario in enumerate(scenario_generator(dataset)):
    if i >= N_SCENARIOS:
        break
    all_rows.extend(scenario_to_rows(scenario))

df = pd.DataFrame(all_rows)
print(f"Loaded {df['scenario_id'].nunique()} scenarios, {df.shape[0]} rows")

# ============================================================
# STEP 5: Fix dtypes (Topic 1, Slide 3)
# ============================================================
df["agent_type"] = df["agent_type"].astype("category")
for col in ["x", "y", "velocity_x", "velocity_y", "heading"]:
    df[col] = df[col].astype("float32")

# ============================================================
# STEP 6: Sort + group by agent (Topic 2, Slide 2)
# ============================================================
df = df.sort_values(["scenario_id", "agent_id", "timestep"])
grouped = df.groupby(["scenario_id", "agent_id"])

DT = 0.1

# ============================================================
# STEP 7: Derived velocity (Topic 2, Slide 3)
# ============================================================
df["dx"] = grouped["x"].diff()
df["dy"] = grouped["y"].diff()
df["velocity_x_derived"] = df["dx"] / DT
df["velocity_y_derived"] = df["dy"] / DT

# ============================================================
# STEP 8: Acceleration + yaw rate (Topic 2, Slide 4)
# ============================================================
grouped = df.groupby(["scenario_id", "agent_id"])
df["speed"] = np.sqrt(df["velocity_x"]**2 + df["velocity_y"]**2)
df["acceleration"] = grouped["speed"].diff() / DT

def wrap_angle(diff):
    return (diff + np.pi) % (2 * np.pi) - np.pi

df["heading_diff"] = wrap_angle(grouped["heading"].diff())
df["yaw_rate"] = df["heading_diff"] / DT

# ============================================================
# STEP 9: Curvature (Topic 2, Slide 5)
# ============================================================
EPS = 1e-3
df["curvature"] = df["yaw_rate"] / (df["speed"] + EPS)
df["path_type"] = np.where(df["curvature"].abs() < 0.02, "straight", "turning")

# ============================================================
# STEP 10: See what we've built
# ============================================================
print("\nShape:", df.shape)
print("\nColumns and dtypes:")
print(df.dtypes)
print("\nSample rows:")
print(df.head(10).to_string())
print("\nNull counts:")
print(df.isnull().sum())

Loaded 5 scenarios, 14275 rows

Shape: (14275, 19)

Columns and dtypes:
scenario_id             object
agent_id                 int64
agent_type            category
timestep                 int64
x                      float32
y                      float32
velocity_x             float32
velocity_y             float32
heading                float32
dx                     float32
dy                     float32
velocity_x_derived     float32
velocity_y_derived     float32
speed                  float32
acceleration           float32
heading_diff           float32
yaw_rate               float32
curvature              float32
path_type               object
dtype: object

Sample rows:
          scenario_id  agent_id agent_type  timestep            x            y  velocity_x  velocity_y   heading   dx   dy  velocity_x_derived  velocity_y_derived  speed  acceleration  heading_diff  yaw_rate  curvature path_type
579  4d82fec943ddaa44       739          1         0 -9170.650391  8076.671875    

In [ ]:
df['agent_id']

,agent_id
579,739
580,739
581,739
582,739
583,739
...,...
12848,3811
12849,3811
12850,3811
12851,3811


In [ ]:
filename = '/content/uncompressed_scenario_validation_validation.tfrecord-00000-of-00150'
dataset = tf.data.TFRecordDataset(filename, compression_type='')
dataset


<TFRecordDatasetV2 element_spec=TensorSpec(shape=(), dtype=tf.string, name=None)>

In [ ]:
def scenario_generator(dataset):
    for raw_record in dataset:
        scenario = scenario_pb2.Scenario()
        scenario.ParseFromString(raw_record.numpy())
        yield scenario



def scenario_to_rows(scenario):
    rows = []
    for track in scenario.tracks:
        for i, state in enumerate(track.states):
            if not state.valid:
                continue
            rows.append({
                "scenario_id": scenario.scenario_id,
                "agent_id": track.id,
                "agent_type": track.object_type,
                "timestep": i,
                "x": state.center_x,
                "y": state.center_y,
                "velocity_x": state.velocity_x,
                "velocity_y": state.velocity_y,
                "heading": state.heading,
            })
    return rows



N_SCENARIOS = 10

all_rows = []
for i, scenario in enumerate(scenario_generator(dataset)):
    if i >= N_SCENARIOS:
        break

    all_rows.extend(scenario_to_rows(scenario))


df = pd.DataFrame(all_rows)
print(f"Loaded {df['scenario_id'].nunique()} scenarios, {df.shape[0]} rows")


Loaded 10 scenarios, 28215 rows


In [ ]:
df

,scenario_id,agent_id,agent_type,timestep,x,y,velocity_x,velocity_y,heading
0,b85e1bd6cc8e74c0,1807,1,0,3578.349121,4986.538086,1.086426,-21.406250,-1.520831
1,b85e1bd6cc8e74c0,1807,1,1,3578.459717,4984.412109,1.105957,-21.259766,-1.518627
2,b85e1bd6cc8e74c0,1807,1,2,3578.583740,4982.276855,1.240234,-21.352539,-1.515478
3,b85e1bd6cc8e74c0,1807,1,3,3578.707520,4980.156738,1.237793,-21.201172,-1.512703
4,b85e1bd6cc8e74c0,1807,1,4,3578.834229,4978.032715,1.267090,-21.240234,-1.512416
...,...,...,...,...,...,...,...,...,...
28210,c95a61d1861f429a,573,1,86,1507.493317,1588.775192,-8.231843,-2.986180,-2.795703
28211,c95a61d1861f429a,573,1,87,1506.664624,1588.471157,-8.363385,-3.055988,-2.793007
28212,c95a61d1861f429a,573,1,88,1505.821527,1588.164318,-8.492949,-3.108458,-2.790310
28213,c95a61d1861f429a,573,1,89,1504.967209,1587.849896,-8.607795,-3.186431,-2.787839
